# Experiment 1 — Class-weight ablation
## XGBoost + GLCM + kernelRadius=5 + cùng patient folds

Mục tiêu: chỉ thay đổi **cơ chế trọng số lớp**:
- `none`: không truyền `sample_weight`;
- `balanced`: tính `compute_class_weight("balanced")` từ **training fold** rồi truyền `sample_weight`.

Các yếu tố còn lại giữ cố định: Development, patient folds, GLCM, XGBoost config và threshold = 0.50.

**Không tải Test trong notebook này.**

In [1]:
import os, json
import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, balanced_accuracy_score
)

RANDOM_STATE = 42
N_SPLITS = 5

# Giữ đúng đường dẫn của notebook hiện tại.
# Nếu dữ liệu thực tế của bạn nằm ở dataset/577p/... thì chỉ sửa biến này.
DATASET_PATH = "dataset/577p/glcm/kernel5/"
TRAIN_PATH = os.path.join(DATASET_PATH, "dataset.train.csv")
TEST_PATH = "dataset/1200p/glcm/kernel5/dataset.test.csv"

BASE_CV_OUTPUT = "output/577p/glcm/kernel5/patient_5fold_cv_saeed_comparison"
CV_SPLITTER = "StratifiedGroupKFold"

XGB_CONFIG = dict(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    min_child_weight=1,
    gamma=0.0,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

OUTPUT_DIR = "output/577p/glcm/kernel5/class_weight_ablation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Cấu hình OOF chính thức dùng nối tiếp cho Experiment 2 và 3.
# Phải đồng bộ với WEIGHTING_MODE trong notebook 03.
OFFICIAL_OOF_MODE = "balanced"  # "balanced" hoặc "none"
OFFICIAL_OOF_PATH = os.path.join(OUTPUT_DIR, "development_oof_predictions.csv")

print("TEST is not loaded in this experiment.")
print("Official OOF mode:", OFFICIAL_OOF_MODE)
print("Official OOF output:", OFFICIAL_OOF_PATH)


TEST is not loaded in this experiment.
Official OOF mode: balanced
Official OOF output: output/577p/glcm/kernel5/class_weight_ablation/development_oof_predictions.csv


## 1. Load Development

In [2]:
train_df = pd.read_csv(TRAIN_PATH)

development = pd.concat([
    train_df.assign(original_split="train"),
], ignore_index=True)

PATIENT_CANDIDATES = ["patient_id", "patient", "subject_id", "case_id"]
patient_col = next((c for c in PATIENT_CANDIDATES if c in development.columns), None)

if patient_col is None:
    raise ValueError("Development CSV thiếu patient_id/subject_id/case_id; không được tiếp tục patient-level CV.")
if "label" not in development.columns:
    raise ValueError("Development CSV thiếu cột label.")

development[patient_col] = development[patient_col].astype(str)
train_df[patient_col] = train_df[patient_col].astype(str)

META = {
    "label", "original_split", patient_col,
    "patient_id", "patient", "subject_id", "case_id",
    "x", "y", "z", "i", "j", "k", "voxel_x", "voxel_y", "voxel_z"
}
numeric_cols = [
    c for c in development.columns
    if c not in META and pd.api.types.is_numeric_dtype(development[c])
]
feature_cols = [c for c in numeric_cols]
if not feature_cols:
    raise ValueError("Không tìm thấy cột GLCM trong Development.")

print("Development shape:", development.shape)
print("Development patients:", development[patient_col].nunique())
print("GLCM features:", len(feature_cols))

Development shape: (653397, 27)
Development patients: 577
GLCM features: 24


## 2. Khóa patient folds bằng StratifiedGroupKFold

Toàn bộ ablation sử dụng **cùng một bộ 5 fold** được tạo bằng `StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)` với `groups=patient_id`. Fold được tạo một lần, lưu ra CSV và tái sử dụng cho cả hai điều kiện `none` và `balanced`.

**Không nạp lại file fold cũ được tạo bởi GroupKFold.** Điều này bảo đảm thực nghiệm này chạy đúng splitter đã chuẩn hóa.


In [3]:
X_tmp = development[feature_cols]
y_tmp = development["label"].astype(int)
g_tmp = development[patient_col].astype(str)

sgkf = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

# Tạo đúng một bộ fold và dùng lại cho mọi điều kiện ablation.
rows = []
for fold, (_, va_idx) in enumerate(sgkf.split(X_tmp, y_tmp, groups=g_tmp), 1):
    for pid in sorted(g_tmp.iloc[va_idx].unique()):
        rows.append({
            patient_col: pid,
            "validation_fold": fold,
            "cv_splitter": CV_SPLITTER,
            "n_splits": N_SPLITS,
            "shuffle": True,
            "random_state": RANDOM_STATE
        })

fold_assignment = pd.DataFrame(rows)
assignment_out = os.path.join(OUTPUT_DIR, "patient_fold_assignment_stratified_groupkfold.csv")
fold_assignment.to_csv(assignment_out, index=False)

if fold_assignment[patient_col].duplicated().any():
    raise RuntimeError("Một patient xuất hiện ở validation của nhiều hơn một fold.")
if set(fold_assignment[patient_col]) != set(development[patient_col].unique()):
    missing = set(development[patient_col].unique()) - set(fold_assignment[patient_col])
    extra = set(fold_assignment[patient_col]) - set(development[patient_col].unique())
    raise RuntimeError(
        f"Fold assignment không khớp Development. missing={len(missing)}, extra={len(extra)}"
    )

patient_to_fold = dict(zip(
    fold_assignment[patient_col],
    fold_assignment["validation_fold"].astype(int)
))
development["validation_fold"] = development[patient_col].map(patient_to_fold)

# Audit số bệnh nhân và tỷ lệ nhãn theo fold.
fold_audit_rows = []
for fold in range(1, N_SPLITS + 1):
    va = development.loc[development["validation_fold"].eq(fold)]
    tr = development.loc[~development["validation_fold"].eq(fold)]
    fold_audit_rows.append({
        "fold": fold,
        "train_patients": tr[patient_col].nunique(),
        "validation_patients": va[patient_col].nunique(),
        "train_voxels": len(tr),
        "validation_voxels": len(va),
        "train_tumor_rate": float(tr["label"].mean()),
        "validation_tumor_rate": float(va["label"].mean())
    })

fold_audit = pd.DataFrame(fold_audit_rows)
fold_audit.to_csv(os.path.join(OUTPUT_DIR, "stratified_groupkfold_audit.csv"), index=False)

print("CV splitter:", CV_SPLITTER)
print("Saved fold assignment:", assignment_out)
display(fold_audit)


CV splitter: StratifiedGroupKFold
Saved fold assignment: output/577p/glcm/kernel5/class_weight_ablation/patient_fold_assignment_stratified_groupkfold.csv


,fold,train_patients,validation_patients,train_voxels,validation_voxels,train_tumor_rate,validation_tumor_rate
0,1,461,116,521577,131820,0.064426,0.060621
1,2,462,115,523132,130265,0.063902,0.062680
2,3,463,114,523605,129792,0.062595,0.067947
3,4,460,117,521845,131552,0.063640,0.063731
4,5,462,115,523429,129968,0.063732,0.063362


## 3. Run two weighting conditions

In [4]:
def binary_metrics(y_true, prob, threshold=0.5):
    y_true = np.asarray(y_true, dtype=int)
    prob = np.asarray(prob, dtype=float)
    pred = (prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    return {
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_true, pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, pred),
        "roc_auc": roc_auc_score(y_true, prob) if np.unique(y_true).size == 2 else np.nan,
        "pr_auc": average_precision_score(y_true, prob) if np.unique(y_true).size == 2 else np.nan,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)
    }

In [5]:
conditions = ["none", "balanced"]
if OFFICIAL_OOF_MODE not in conditions:
    raise ValueError("OFFICIAL_OOF_MODE phải là 'balanced' hoặc 'none'.")

fold_rows = []
oof_parts = []

for condition in conditions:
    for fold in range(1, N_SPLITS + 1):
        va_mask = development["validation_fold"].eq(fold)
        tr_mask = ~va_mask

        tr = development.loc[tr_mask].copy()
        va = development.loc[va_mask].copy()

        tr_pat = set(tr[patient_col].unique())
        va_pat = set(va[patient_col].unique())
        if tr_pat & va_pat:
            raise RuntimeError(f"{condition}, fold {fold}: patient leakage")

        Xtr = tr[feature_cols]
        ytr = tr["label"].astype(int)
        Xva = va[feature_cols]
        yva = va["label"].astype(int)

        fit_kwargs = {}
        w0 = w1 = np.nan

        if condition == "balanced":
            classes = np.array([0, 1])
            weights = compute_class_weight(class_weight="balanced", classes=classes, y=ytr)
            cw = dict(zip(classes, weights))
            fit_kwargs["sample_weight"] = ytr.map(cw).to_numpy()
            w0, w1 = float(cw[0]), float(cw[1])

        model = XGBClassifier(**XGB_CONFIG)
        model.fit(Xtr, ytr, verbose=False, **fit_kwargs)

        prob = model.predict_proba(Xva)[:, 1]
        m = binary_metrics(yva, prob, threshold=0.5)

        fold_rows.append({
            "condition": condition,
            "cv_splitter": CV_SPLITTER,
            "fold": fold,
            "train_patients": tr[patient_col].nunique(),
            "validation_patients": va[patient_col].nunique(),
            "train_voxels": len(tr),
            "validation_voxels": len(va),
            "class_weight_0": w0,
            "class_weight_1": w1,
            **m
        })

        part = va[[patient_col, "label"]].copy()
        for c in ["x","y","z","i","j","k","voxel_x","voxel_y","voxel_z"]:
            if c in va.columns:
                part[c] = va[c].values
        part["condition"] = condition
        part["cv_splitter"] = CV_SPLITTER
        part["fold"] = fold
        part["prob_tumor"] = prob
        part["pred_t05"] = (prob >= 0.5).astype(int)
        oof_parts.append(part)

        print(condition, "fold", fold, "F1", round(m["f1"], 4), "PR-AUC", round(m["pr_auc"], 4))

fold_results = pd.DataFrame(fold_rows)
oof = pd.concat(oof_parts, ignore_index=True)

# Lưu toàn bộ kết quả ablation (cả none và balanced).
fold_results.to_csv(
    os.path.join(OUTPUT_DIR, "class_weight_ablation_fold_metrics.csv"),
    index=False
)
oof.to_csv(
    os.path.join(OUTPUT_DIR, "class_weight_ablation_oof_predictions.csv"),
    index=False
)

# Lưu riêng OOF theo từng condition để audit/reuse khi cần.
for condition in conditions:
    condition_oof = oof.loc[oof["condition"].eq(condition)].copy()
    condition_oof.to_csv(
        os.path.join(OUTPUT_DIR, f"development_oof_predictions_{condition}.csv"),
        index=False
    )

# Xuất OOF CHÍNH THỨC dùng cho notebook 02 và 03.
official_oof = oof.loc[oof["condition"].eq(OFFICIAL_OOF_MODE)].copy()

# Kiểm tra mỗi voxel Development chỉ xuất hiện đúng một lần trong OOF chính thức.
if len(official_oof) != len(development):
    raise RuntimeError(
        f"Official OOF rows ({len(official_oof)}) != Development rows ({len(development)})."
    )

if official_oof[patient_col].nunique() != development[patient_col].nunique():
    raise RuntimeError(
        "Official OOF không bao phủ đúng toàn bộ bệnh nhân Development."
    )

expected_patients = set(development[patient_col].astype(str).unique())
oof_patients = set(official_oof[patient_col].astype(str).unique())
if expected_patients != oof_patients:
    raise RuntimeError(
        f"Official OOF patient mismatch: missing={len(expected_patients - oof_patients)}, "
        f"extra={len(oof_patients - expected_patients)}"
    )

official_oof.to_csv(OFFICIAL_OOF_PATH, index=False)

print("\nSaved official Development OOF:")
print("  mode:", OFFICIAL_OOF_MODE)
print("  path:", OFFICIAL_OOF_PATH)
print("  rows:", len(official_oof))
print("  patients:", official_oof[patient_col].nunique())
display(fold_results)


none fold 1 F1 0.7331 PR-AUC 0.828
none fold 2 F1 0.7116 PR-AUC 0.7868
none fold 3 F1 0.7244 PR-AUC 0.8118
none fold 4 F1 0.7086 PR-AUC 0.79
none fold 5 F1 0.6973 PR-AUC 0.8004
balanced fold 1 F1 0.6359 PR-AUC 0.8123
balanced fold 2 F1 0.5187 PR-AUC 0.7754
balanced fold 3 F1 0.5803 PR-AUC 0.8088
balanced fold 4 F1 0.5617 PR-AUC 0.7844
balanced fold 5 F1 0.6206 PR-AUC 0.7965

Saved official Development OOF:
  mode: balanced
  path: output/577p/glcm/kernel5/class_weight_ablation/development_oof_predictions.csv
  rows: 653397
  patients: 577


,condition,cv_splitter,fold,train_patients,validation_patients,train_voxels,validation_voxels,class_weight_0,class_weight_1,accuracy,...,recall,specificity,f1,mcc,roc_auc,pr_auc,tn,fp,fn,tp
0,none,StratifiedGroupKFold,1,461,116,521577,131820,NaN,NaN,0.971211,...,0.652109,0.991803,0.733066,0.724351,0.979004,0.828033,122814,1015,2780,5211
1,none,StratifiedGroupKFold,2,462,115,523132,130265,NaN,NaN,0.966422,...,0.660992,0.986847,0.711630,0.696220,0.964164,0.786756,120494,1606,2768,5397
2,none,StratifiedGroupKFold,3,463,114,523605,129792,NaN,NaN,0.965345,...,0.670144,0.986865,0.724353,0.708635,0.971829,0.811753,119384,1589,2909,5910
3,none,StratifiedGroupKFold,4,460,117,521845,131552,NaN,NaN,0.966173,...,0.645277,0.988016,0.708579,0.694579,0.967073,0.789950,121692,1476,2974,5410
4,none,StratifiedGroupKFold,5,462,115,523429,129968,NaN,NaN,0.966861,...,0.602429,0.991514,0.697308,0.689854,0.967291,0.800409,120700,1033,3274,4961
5,balanced,StratifiedGroupKFold,1,461,116,521577,131820,0.534431,7.760870,0.937187,...,0.904893,0.939271,0.635916,0.639009,0.977940,0.812343,116309,7520,760,7231
6,balanced,StratifiedGroupKFold,2,462,115,523132,130265,0.534132,7.824524,0.894976,...,0.902756,0.894455,0.518664,0.533213,0.963629,0.775416,109213,12887,794,7371
7,balanced,StratifiedGroupKFold,3,463,114,523605,129792,0.533387,7.987872,0.910018,...,0.915637,0.909608,0.580330,0.587369,0.972200,0.808763,110038,10935,744,8075
8,balanced,StratifiedGroupKFold,4,460,117,521845,131552,0.533982,7.856745,0.910803,...,0.896708,0.911763,0.561674,0.569557,0.966538,0.784447,112300,10868,866,7518
9,balanced,StratifiedGroupKFold,5,462,115,523429,129968,0.534035,7.845394,0.931529,...,0.883910,0.934751,0.620625,0.620205,0.967611,0.796545,113790,7943,956,7279


### OOF output dùng cho Experiment 2 và 3

`development_oof_predictions.csv` chỉ chứa OOF của cấu hình `OFFICIAL_OOF_MODE` đã khóa. OOF được sinh từ **5-fold StratifiedGroupKFold theo patient_id**. Mỗi bệnh nhân Development chỉ được dự đoán ở fold mà bệnh nhân đó thuộc validation, vì vậy file này có thể dùng tiếp cho per-patient bootstrap và chọn threshold mà không dùng Test.


## 4. Mean ± SD and paired fold deltas

In [6]:
metric_cols = [
    "accuracy","balanced_accuracy","precision","recall","specificity",
    "f1","mcc","roc_auc","pr_auc"
]

summary_rows = []
for condition, g in fold_results.groupby("condition"):
    row = {"condition": condition}
    for metric in metric_cols:
        row[f"{metric}_mean"] = g[metric].mean()
        row[f"{metric}_sd"] = g[metric].std(ddof=1)
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary.to_csv(os.path.join(OUTPUT_DIR, "class_weight_ablation_summary.csv"), index=False)

wide = fold_results.pivot(index="fold", columns="condition", values=metric_cols)
delta_rows = []
for fold in sorted(fold_results["fold"].unique()):
    row = {"fold": fold}
    for metric in metric_cols:
        row[f"delta_balanced_minus_none_{metric}"] = (
            wide.loc[fold, (metric, "balanced")] - wide.loc[fold, (metric, "none")]
        )
    delta_rows.append(row)

paired_delta = pd.DataFrame(delta_rows)
paired_delta.to_csv(os.path.join(OUTPUT_DIR, "class_weight_ablation_paired_deltas.csv"), index=False)

display(summary)
display(paired_delta)

,condition,accuracy_mean,accuracy_sd,balanced_accuracy_mean,balanced_accuracy_sd,precision_mean,precision_sd,recall_mean,recall_sd,specificity_mean,specificity_sd,f1_mean,f1_sd,mcc_mean,mcc_sd,roc_auc_mean,roc_auc_sd,pr_auc_mean,pr_auc_sd
0,balanced,0.916903,0.017254,0.909375,0.008866,0.433185,0.051826,0.900781,0.011648,0.917970,0.018686,0.583442,0.046946,0.589871,0.041724,0.969584,0.005597,0.795503,0.015702
1,none,0.967202,0.002308,0.817600,0.012291,0.801812,0.028824,0.646190,0.026191,0.989009,0.002467,0.714987,0.013959,0.702728,0.013937,0.969872,0.005795,0.803380,0.016907


,fold,delta_balanced_minus_none_accuracy,delta_balanced_minus_none_balanced_accuracy,delta_balanced_minus_none_precision,delta_balanced_minus_none_recall,delta_balanced_minus_none_specificity,delta_balanced_minus_none_f1,delta_balanced_minus_none_mcc,delta_balanced_minus_none_roc_auc,delta_balanced_minus_none_pr_auc
0,1,-0.034024,0.100126,-0.346770,0.252784,-0.052532,-0.097150,-0.085342,-0.001064,-0.015690
1,2,-0.071447,0.074686,-0.406813,0.241764,-0.092391,-0.192965,-0.163007,-0.000535,-0.011340
2,3,-0.055327,0.084118,-0.363329,0.245493,-0.077257,-0.144024,-0.121266,0.000371,-0.002991
3,4,-0.055370,0.087589,-0.376754,0.251431,-0.076254,-0.146905,-0.125022,-0.000535,-0.005504
4,5,-0.035332,0.112359,-0.349472,0.281481,-0.056764,-0.076683,-0.069649,0.000320,-0.003863


## Interpretation rule

Không chọn cấu hình bằng Test. Nếu cần khóa weighting cho final model, lựa chọn phải dựa trên kết quả Development/CV.
Nên ưu tiên PR-AUC và F1-score, đồng thời theo dõi Precision, Recall và MCC.